# Sensor Data Cleaner & Visualizer

A beginner-friendly walk through loading messy real-world sensor data,
cleaning it, visualizing it, and saving the results.

**Dataset:** [UCI Air Quality](https://archive.ics.uci.edu/dataset/387/air+quality)

We will first build a small *fake* file with problems we already know about,
practice cleaning on it, and only then point the same code at the real file.
That way, when something looks wrong, you'll know whether it's the data or your code.

Run the cells **top to bottom** (Shift+Enter runs a cell and moves to the next one).

## 1. Setup

We bring in three libraries. Think of these as toolboxes we unpack once at the top.

- **pandas** — works with tables of data (rows and columns), like a spreadsheet you control with code.
- **numpy** — fast math on numbers; we mainly use it for its special "missing value" marker, `np.nan`.
- **matplotlib** — draws charts so we can *see* the data instead of just reading numbers.

In [ ]:
import os                      # for creating folders and file paths

import pandas as pd              # 'pd' is the conventional nickname everyone uses
import numpy as np               # 'np' likewise
import matplotlib.pyplot as plt  # 'plt' is the drawing surface for charts

**What just happened:** `import ... as ...` loads a library and gives it a short
nickname so we don't type the full name every time. Nothing is printed — that's normal.
If this cell runs without an error, all three toolboxes are ready.

## 2. Make practice data

Before touching the real file, we generate a **fake CSV that copies the UCI layout**
and bake in the exact problems we'll need to fix:

- the `-200` value that the sensors write when a reading failed (a "sentinel"),
- a couple of **duplicate** rows,
- a few **outlier spikes** (impossibly large readings),
- **two empty trailing columns** and some **empty rows** at the very end,
- numbers written with a **comma** as the decimal point (`2,5` means 2.5) and fields
  separated by **semicolons** — exactly like the European-formatted UCI file.

Because we created these problems on purpose, we know precisely what a correct cleaning
step should remove.

In [ ]:
# np.random.seed makes the "random" numbers come out the same every time you run this,
# so your results match the explanations below.
np.random.seed(42)

n_rows = 200

# A timestamp for each row, one hour apart, just like hourly sensor logs.
start = pd.Timestamp("2004-03-10 18:00:00")
timestamps = [start + pd.Timedelta(hours=i) for i in range(n_rows)]

# Three plausible sensor readings. normal(center, spread, count) draws 'count' numbers
# scattered around 'center'.
co = np.random.normal(2.0, 0.5, n_rows)    # carbon monoxide
nox = np.random.normal(150, 40, n_rows)    # nitrogen oxides
temp = np.random.normal(15, 5, n_rows)     # temperature in Celsius

# Problem 1: sensor-failure sentinels (-200) scattered through two of the columns.
for i in [5, 23, 24, 80, 81, 82, 150]:
    co[i] = -200
for i in [10, 45, 120]:
    nox[i] = -200

# Problem 2: outlier spikes — physically impossible jumps.
co[100] = 35.0
nox[60] = 1200.0

**What just happened:** we made three columns of believable numbers, then *damaged*
them on purpose. `np.random.normal` returns an array of random numbers; assigning
`co[5] = -200` overwrites one reading with the failure sentinel. Nothing is saved to disk yet.

In [ ]:
# Now write those arrays to a text file shaped exactly like the UCI download.
# We build the file line by line so you can see every quirk being created.

header = "Date;Time;CO_GT;NOx_GT;Temp;;"   # note the TWO trailing ';' -> two empty columns
lines = [header]

for i in range(n_rows):
    date_str = timestamps[i].strftime("%d/%m/%Y")   # DD/MM/YYYY, e.g. 10/03/2004
    time_str = timestamps[i].strftime("%H.%M.%S")   # HH.MM.SS, e.g. 18.00.00

    # Format each number to one decimal, then swap the '.' for a ',' (European style).
    co_str = f"{co[i]:.1f}".replace(".", ",")
    nox_str = f"{nox[i]:.1f}".replace(".", ",")
    temp_str = f"{temp[i]:.1f}".replace(".", ",")

    lines.append(f"{date_str};{time_str};{co_str};{nox_str};{temp_str};;")

# Problem: a couple of duplicate rows (exact copies of earlier rows).
lines.append(lines[1])
lines.append(lines[2])

# Problem: empty rows at the very end (just separators, no values).
lines.append(";;;;;;")
lines.append(";;;;;;")

# Make a folder for raw input files and write the CSV into it.
os.makedirs("data/raw", exist_ok=True)
practice_path = "data/raw/practice_air_quality.csv"
with open(practice_path, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

print("Wrote", len(lines), "lines to", practice_path)

**What just happened:**

- `strftime` turns a timestamp into a formatted text string (the codes `%d`, `%m`, `%Y`,
  `%H`, etc. are placeholders for day, month, year, hour…).
- `f"{co[i]:.1f}"` is an *f-string*: it drops the value of `co[i]` into the text, rounded
  to one decimal. `.replace(".", ",")` then converts `2.5` into `2,5`.
- `os.makedirs(..., exist_ok=True)` creates the `data/raw/` folder, and `exist_ok=True`
  means "don't complain if it already exists."
- `open(path, "w")` opens a file for **w**riting; `"\n".join(lines)` glues all the lines
  together with newlines between them.

We now have a messy file on disk that looks just like the real UCI download.

## 3. Load the data

Now we read that file into a pandas **DataFrame** — a table in memory we can inspect and edit.

The three quirks the reader must handle:
- `sep=';'` — fields are split by semicolons, not commas.
- `decimal=','` — a comma is the decimal point, so `2,5` is read as the number 2.5.

> **Switching to the real file later:** download `AirQualityUCI.csv` from the UCI page,
> drop it in `data/raw/`, and change `data_path` below to point at it. The cleaning code
> that follows is written to work for both files.

In [ ]:
data_path = "data/raw/practice_air_quality.csv"   # change to the real file when ready

raw_df = pd.read_csv(data_path, sep=";", decimal=",")

raw_df.head()   # show the first 5 rows

**What just happened:** `pd.read_csv(...)` reads a text file and *returns* a DataFrame,
which we store in `raw_df` ("raw" = untouched, as-loaded). `.head()` returns the first 5 rows
so we can eyeball the table. Notice the two `Unnamed` columns on the right — those are the
empty trailing columns we'll drop later.

In [ ]:
raw_df.info()   # a summary: column names, how many non-empty values, and each column's type

**What just happened:** `.info()` prints a summary rather than returning a table. The key
things to read here:
- **column names and count** — confirm the two `Unnamed` columns exist,
- **Non-Null Count** — how many real values each column has (empty cells don't count),
- **Dtype** — the data *type*: `float64` means numbers, `object` usually means text.

`Date` and `Time` show up as `object` (text) — we'll convert them to a real timestamp soon.

In [ ]:
raw_df.describe()   # quick statistics for the numeric columns

**What just happened:** `.describe()` returns a small table of statistics (count, mean,
min, max, …) for each numeric column. **Look at the `min` row:** it's `-200`. That's the
giant clue that missing values are hiding as `-200` and dragging the averages down. A real
air-quality reading is never `-200`.

## 4. Inspect the mess

Two different kinds of "missing" exist in this file:
1. **truly blank** cells (the empty rows/columns) — pandas already shows these as `NaN`,
2. the **`-200` sentinel** — pandas thinks these are normal numbers, so we have to find them.

In [ ]:
raw_df.isna().sum()   # count the already-blank (NaN) cells in each column

**What just happened:** `.isna()` returns a same-shaped table of `True`/`False`
(`True` where a cell is blank). Summing a column of True/False counts the `True`s
(True = 1, False = 0), so `.sum()` gives the number of blanks per column. The two
`Unnamed` columns are entirely blank — that's our empty trailing columns.

In [ ]:
# The -200 sentinels are NOT blank yet, so isna() misses them. Count them directly:
(raw_df == -200).sum()

**What just happened:** `raw_df == -200` builds a True/False table marking every cell
equal to `-200`, and `.sum()` counts them per column. These are the failed readings disguised
as numbers. **This is the core problem of the dataset:** until we turn `-200` into a real
"missing" marker, every average, plot, and calculation will be wrong.

## 5. Clean it

We'll fix the data in clear, one-idea-at-a-time steps, always working on a **copy** called
`clean_df` so the original `raw_df` stays available for comparison.

The order matters, so here's the plan:
1. copy the data,
2. drop the empty trailing **columns** and empty **rows**,
3. remove duplicate rows,
4. combine `Date` + `Time` into one real timestamp and use it as the index,
5. turn `-200` into the proper missing marker `NaN`,
6. fill the gaps (we'll compare two methods),
7. flag and fix outlier spikes.

In [ ]:
clean_df = raw_df.copy()   # work on a copy; never destroy your original load
clean_df.shape             # (rows, columns) before cleaning

**What just happened:** `.copy()` returns an independent duplicate, so edits to `clean_df`
never touch `raw_df`. `.shape` is an attribute (no parentheses) giving `(rows, columns)`.
Note these numbers so you can see them shrink as we remove junk.

In [ ]:
# Step 2a: drop columns that are entirely empty (the two trailing 'Unnamed' ones).
clean_df = clean_df.dropna(axis=1, how="all")

# Step 2b: drop rows that are entirely empty (the blank rows at the end).
clean_df = clean_df.dropna(axis=0, how="all")

clean_df.shape

**What just happened:** `.dropna()` removes missing data and *returns* a trimmed table.
- `axis=1` means operate on **columns**, `axis=0` means **rows**.
- `how="all"` means "only drop it if *every* value is missing" — so we delete the fully-empty
  columns and rows without accidentally losing rows that have just one gap.

The column count and row count should both have dropped.

In [ ]:
# Step 3: remove exact duplicate rows (we planted two earlier).
before = len(clean_df)
clean_df = clean_df.drop_duplicates()
print("Removed", before - len(clean_df), "duplicate row(s)")

**What just happened:** `.drop_duplicates()` returns the table with repeated rows removed,
keeping the first occurrence. `len(clean_df)` counts rows, so subtracting tells us how many
duplicates went away. We do this *before* making the timestamp, while the duplicate rows are
still perfectly identical.

In [ ]:
# Step 4: combine the separate Date and Time text columns into ONE real timestamp.
# We glue them with a space, then tell pandas the exact format the text is in.
clean_df["datetime"] = pd.to_datetime(
    clean_df["Date"] + " " + clean_df["Time"],
    format="%d/%m/%Y %H.%M.%S",
)

# We no longer need the original text columns.
clean_df = clean_df.drop(columns=["Date", "Time"])

# Use the timestamp as the row index (the row labels). This makes time-based
# plotting and sorting effortless later.
clean_df = clean_df.set_index("datetime")

clean_df.head()

**What just happened:**
- `clean_df["Date"] + " " + clean_df["Time"]` joins the two text columns into one like
  `"10/03/2004 18.00.00"`.
- `pd.to_datetime(..., format=...)` reads that text as a real timestamp. The `format` string
  spells out the layout: `%d/%m/%Y` for `DD/MM/YYYY` and `%H.%M.%S` for `HH.MM.SS`. Giving the
  exact format makes parsing fast and unambiguous (so it never confuses day and month).
- `.drop(columns=[...])` removes columns we're done with.
- `.set_index("datetime")` promotes the timestamp to be the row labels. From now on every row
  is identified by *when* it was recorded.

In [ ]:
# Remember every remaining column is a sensor now (the timestamp became the index).
# Capturing the names in a variable keeps the rest of the notebook generic, so the
# SAME code works on the real UCI file even though its columns are named differently.
sensor_cols = clean_df.columns.tolist()
sensor_cols

**What just happened:** `.columns` lists the column names; `.tolist()` turns that into a
plain Python list. We'll reuse `sensor_cols` for filling, outlier checks, and plotting instead
of hard-coding `"CO_GT"` everywhere.

In [ ]:
# Step 5: replace the -200 sentinel with NaN, the real "missing" marker.
# np.nan is numpy's standard stand-in for "no value here".
clean_df = clean_df.replace(-200, np.nan)

clean_df.isna().sum()   # now the former -200 cells are counted as missing

**What just happened:** `.replace(-200, np.nan)` returns a copy where every `-200` becomes
`NaN`. Now `.isna().sum()` finally counts the failed readings as missing. Compare this to the
count from section 4 — the numbers should match the `-200` counts we found earlier. The data is
now *honest* about what it doesn't know.

### Filling the gaps: two methods

We have holes to fill. Two beginner-friendly approaches:

- **`.interpolate()`** — draws a straight line between the known values on either side of a gap
  and reads off the in-between values. Great for things that change *smoothly over time*, like
  temperature or a slowly drifting gas reading.
- **`.ffill()`** ("forward fill") — copies the *last known* value forward into the blanks. Good
  for values that hold steady until they change, like a thermostat setting or a status flag.

> **Note on `fillna(method='ffill')`:** older tutorials write forward-fill as
> `clean_df.fillna(method='ffill')`. That spelling was **removed in pandas 3.0** (the version
> you have), so it now raises an error. The modern, equivalent call is simply `clean_df.ffill()`,
> which is what we use below.

In [ ]:
# Compare the two methods on a copy each (we don't commit yet — just look).
demo_interp = clean_df.interpolate(method="linear")
demo_ffill = clean_df.ffill()

# Rows 23 and 24 had -200 in the CO column, so they're a good place to see the difference.
comparison = pd.DataFrame({
    "with gaps": clean_df["CO_GT"],
    "interpolate": demo_interp["CO_GT"],
    "ffill": demo_ffill["CO_GT"],
})
comparison.iloc[22:26]   # rows 22..25 by position

**What just happened:** we filled two throwaway copies and lined up the CO column three ways.
In the gap rows you can see **interpolate** lands on a value *between* the neighbours, while
**ffill** just repeats the value from above. `.iloc[22:26]` selects rows 22 to 25 *by position*
(as opposed to `.loc`, which selects by label).

In [ ]:
# Commit to interpolation for these smoothly-changing sensors.
clean_df = clean_df.interpolate(method="linear")

# Interpolation can't fill a gap at the very START (no earlier value to lean on),
# so back-fill any leftover leading gaps with the first known value.
clean_df = clean_df.bfill()

clean_df.isna().sum()   # should be all zeros now

**What just happened:** we applied linear interpolation for real this time, then used
`.bfill()` ("backward fill") to mop up any gap at the top of a column that interpolation
couldn't reach. The missing-value counts should all be `0` now.

In [ ]:
# Step 7: flag outlier spikes. A simple, common rule: anything more than 3 standard
# deviations from a column's mean is suspiciously far out. We replace those with NaN
# and interpolate again, just like a missing value.
for col in sensor_cols:
    mean = clean_df[col].mean()
    std = clean_df[col].std()
    lower, upper = mean - 3 * std, mean + 3 * std

    is_outlier = (clean_df[col] < lower) | (clean_df[col] > upper)
    count = int(is_outlier.sum())
    if count:
        print(f"{col}: flagged {count} outlier(s) outside [{lower:.1f}, {upper:.1f}]")

    clean_df.loc[is_outlier, col] = np.nan

# Fill the holes the outlier removal just made.
clean_df = clean_df.interpolate(method="linear").bfill()
print("Done. Remaining missing values:", int(clean_df.isna().sum().sum()))

**What just happened:**
- `.mean()` and `.std()` return a column's average and its *standard deviation* (a measure of
  how spread out the values are). Roughly: almost all normal readings sit within 3 standard
  deviations of the mean, so anything beyond that is a strong outlier candidate.
- `(clean_df[col] < lower) | (clean_df[col] > upper)` builds a True/False mask of the spikes
  (`|` means "or").
- `clean_df.loc[is_outlier, col] = np.nan` sets just those flagged cells to missing, then we
  interpolate them away — the same trick we used for the `-200`s.

This caught the impossible `35` and `1200` spikes we planted. `.sum().sum()` adds up the
per-column counts and then totals them into a single number.

## 6. Visualize

Numbers in a table are hard to judge; a picture makes problems obvious. We'll draw:
1. each sensor over time, and
2. a **before-vs-after** overlay for one sensor so you can watch the `-200` dips disappear.

In [ ]:
# One stacked chart per sensor, sharing the same time axis along the bottom.
fig_sensors, axes = plt.subplots(
    len(sensor_cols), 1, figsize=(10, 7), sharex=True
)

for ax, col in zip(axes, sensor_cols):
    ax.plot(clean_df.index, clean_df[col])
    ax.set_ylabel(col)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("time")
fig_sensors.suptitle("Cleaned sensor readings over time")
fig_sensors.tight_layout()
plt.show()

**What just happened:**
- `plt.subplots(rows, cols)` *returns two things*: the overall `fig` (the whole picture) and the
  `axes` (the individual mini-charts). `sharex=True` lines up their time axes.
- `zip(axes, sensor_cols)` pairs each mini-chart with one sensor name so the loop draws one per
  panel. `ax.plot(x, y)` draws a line; here x is the timestamp index and y is the sensor column.
- `plt.show()` displays the figure. The lines should look smooth now — no `-200` cliffs.

In [ ]:
# Before-vs-after for the first sensor. We rebuild a "before" version straight from the
# raw load so the -200 dips and the spike are visible, then overlay the cleaned version.
first = sensor_cols[0]

before = raw_df.copy()
before["datetime"] = pd.to_datetime(
    before["Date"] + " " + before["Time"], format="%d/%m/%Y %H.%M.%S", errors="coerce"
)
before = before.dropna(subset=["datetime"]).set_index("datetime")

fig_compare, ax = plt.subplots(figsize=(10, 4))
ax.plot(before.index, before[first], label="before (raw, -200 present)", alpha=0.5)
ax.plot(clean_df.index, clean_df[first], label="after (cleaned)", linewidth=2)
ax.set_title(f"{first}: before vs after cleaning")
ax.set_xlabel("time")
ax.set_ylabel(first)
ax.legend()
fig_compare.tight_layout()
plt.show()

**What just happened:** we reconstructed a quick "before" table from `raw_df` (giving it the
same timestamp index so the two lines align), then plotted both on one chart. `errors="coerce"`
tells `to_datetime` to turn any unparseable rows (the empty ones) into `NaT` instead of crashing,
and we drop those. The faint "before" line plunges to `-200` at the failed readings; the bold
"after" line stays smooth. `ax.legend()` draws the little key explaining which line is which.

## 7. Save results

Finally, write the cleaned table and the charts to disk so you can use them elsewhere.

In [ ]:
# Save the cleaned data as a normal comma-separated CSV (the standard, portable format).
os.makedirs("data/processed", exist_ok=True)
output_csv = "data/processed/cleaned_air_quality.csv"
clean_df.to_csv(output_csv)   # the datetime index is saved as the first column
print("Saved cleaned data ->", output_csv)

**What just happened:** `.to_csv(path)` writes the DataFrame to a file. Because our index is
the timestamp, it becomes the first column of the output. Unlike the European source file, this
one uses plain commas and dots — the universal default — so other tools read it without fuss.

In [ ]:
# Save the two figures as PNG image files.
os.makedirs("plots", exist_ok=True)
fig_sensors.savefig("plots/sensors_over_time.png", dpi=150, bbox_inches="tight")
fig_compare.savefig("plots/before_after.png", dpi=150, bbox_inches="tight")
print("Saved plots/ -> sensors_over_time.png, before_after.png")

**What just happened:** `fig.savefig(path)` writes a figure to an image file. `dpi=150` sets
the resolution (dots per inch — higher is sharper) and `bbox_inches="tight"` trims off extra white
border. We saved the figure objects (`fig_sensors`, `fig_compare`) from earlier, which is why we
could write them out here.

**You did it.** You loaded a deliberately messy file, made its missing values honest, filled and
de-spiked it, visualized the improvement, and exported clean results.

## Good habits for later

### Add a `.gitignore`

When you put this project under git, create a file named **`.gitignore`** in the project root
with the lines below. Git will then ignore these — keeping secrets, bulky raw data, and
auto-generated junk out of your commits:

```gitignore
# secrets — never commit these
.env

# raw data dumps — often large, and re-downloadable
data/raw/

# Python's auto-generated cache
__pycache__/
```

### When you later pull *live* data (API keys)

This project is local-only, so there are no secrets yet. But the moment you fetch live data
from an online service, you'll get an **API key** — a password that proves it's you. Handle it
like a password:

1. Put it in a file called **`.env`** (which the `.gitignore` above keeps out of git):

   ```
   AIR_QUALITY_API_KEY=your_secret_key_here
   ```

2. Load it in Python with the **python-dotenv** package and read it with `os.getenv`:

   ```python
   from dotenv import load_dotenv   # pip install python-dotenv
   import os

   load_dotenv()                              # reads the .env file into the environment
   api_key = os.getenv("AIR_QUALITY_API_KEY") # fetch the value by name
   ```

**Never** hardcode a key directly in your code, and **never** `print()` it or paste it into a
notebook cell — anything committed to git or shared in a notebook output can be copied by anyone
who sees it. The `.env` + `os.getenv` pattern keeps the key on your machine only.